<a href="https://colab.research.google.com/github/nageshnnazare/cuda-bench/blob/main/colab/cuda_bench_matmul_launcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [63]:
!nvidia-smi

Tue Nov 24 20:36:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [64]:
!rm -rf /content/cuda-bench
%cd '/content'
!git clone https://github.com/nageshnnazare/cuda-bench.git '/content/cuda-bench/'
%cd '/content/cuda-bench'
!chmod +x scripts/profile.sh

/content
Cloning into '/content/cuda-bench'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 61 (delta 19), reused 47 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 22.88 KiB | 22.88 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/cuda-bench


In [65]:
!make matmul

mkdir -p bin
nvcc -O3 -std=c++17 -arch=sm_75 -Iinclude/ -lineinfo -Xptxas=-v,-warn-spills src/matmul/01_naive.cu -o bin/matmul_01_naive  2> bin/matmul_01_naive.ptxas.log || { cat bin/matmul_01_naive.ptxas.log; exit 1; }; grep -E "warning|spill|registers" bin/matmul_01_naive.ptxas.log | grep -v " 0 bytes spill" || true
ptxas info    : Used 52 registers, used 0 barriers, 392 bytes cmem[0]


In [61]:
!make run

mkdir -p reports
== bin/matmul_01_naive
Device: Tesla T4 (sm_75, 40 SMs, 15.6 GB)
  [ ok ] v0 naive vs CPU              max_abs_err=1.431e-05

v0 naive                    278.269 ms    0.0617 TFLOP/s


In [66]:
!make profile matmul

./scripts/profile.sh matmul_01_naive
== matmul_01_naive
### full report: ncu --set full --import-source yes
==PROF== Connected to process 24165 (/content/cuda-bench/bin/matmul_01_naive)
==PROF== Profiling "sgemm_naive" - 0: 0%
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
....50%....100% - 33 passes
Device: Tesla T4 (sm_75, 40 SMs, 15.6 GB)

v0 naive                  41638.891 ms    0.0004 TFLOP/s
==PROF== Disconnected from process 24165
==PROF== Report: /content/cuda-bench/reports/20261124T203613Z_Tesla-T4_clock-base/matmul_01_naive.ncu-rep
### metric table: ncu --metrics (the raw CSV is written to matmul_01_naive.csv)
| kernel | time (us) | SM % | Mem % | DRAM rd (MB) | DRAM wr (MB) | occupancy % | regs | sectors/req | smem conflicts 

In [47]:
%cd '/content/'
!rm -rf cuda-bench.zip
!zip -r 'cuda-bench.zip' 'cuda-bench/'

/content
  adding: cuda-bench/ (stored 0%)
  adding: cuda-bench/bin/ (stored 0%)
  adding: cuda-bench/bin/matmul_01_naive (deflated 69%)
  adding: cuda-bench/bin/matmul_01_naive.ptxas.log (deflated 45%)
  adding: cuda-bench/.gitignore (deflated 5%)
  adding: cuda-bench/src/ (stored 0%)
  adding: cuda-bench/src/matmul/ (stored 0%)
  adding: cuda-bench/src/matmul/01_naive.cu (deflated 63%)
  adding: cuda-bench/colab/ (stored 0%)
  adding: cuda-bench/colab/cuda-bench.ipynb (deflated 14%)
  adding: cuda-bench/colab/cuda_bench_matmul_launcher.ipynb (deflated 73%)
  adding: cuda-bench/reports/ (stored 0%)
  adding: cuda-bench/reports/timings.csv (deflated 7%)
  adding: cuda-bench/include/ (stored 0%)
  adding: cuda-bench/include/common/ (stored 0%)
  adding: cuda-bench/include/common/helpers.cuh (deflated 66%)
  adding: cuda-bench/.git/ (stored 0%)
  adding: cuda-bench/.git/branches/ (stored 0%)
  adding: cuda-bench/.git/logs/ (stored 0%)
  adding: cuda-bench/.git/logs/HEAD (deflated 41%)
  

In [46]:
from google.colab import files
files.download('/content/cuda-bench.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>